# scisTest example

This notebook demonstrates the SCIS test with single split for a p value and multi split for an aggregated p value.

Install the package from the repository root before running the notebook:

```bash
python -m pip install -e .
```

In [1]:
from scistest import (
    DirectionConfig,
    GeneratorConfig,
    HazardConfig,
    RandomForestDirectionConfig,
    SciTestConfig,
    aggregated_scis_test,
    scis_test,
)
from scistest.simulation import simulate_survival_data

## Simulate right-censored survival data

The example uses a nonzero effect so that $U$ is not conditionally independent of $X$ given $Z$.

In [2]:
data = simulate_survival_data(n=400, effect=0.8, random_state=8)
{
    "x_shape": data.x.shape,
    "z_shape": data.z.shape,
    "event_fraction": float(data.event.mean()),
}

{'x_shape': (400, 1), 'z_shape': (400, 3), 'event_fraction': 0.5375}

## Default DNN direction estimator

Passing `DirectionConfig` is optional: `scis_test` uses this DNN estimator by default. The smaller settings below keep the example quick.

In [3]:
hazard = HazardConfig(hidden_dims=(16,), epochs=15)
dnn_direction = DirectionConfig(hidden_dims=(16,), epochs=15)
generator = GeneratorConfig(epochs=15, hidden_dim=32, noise_dim=32)
test = SciTestConfig(integration_points=20, generator_draws=20)

dnn_result = scis_test(
    data.x,
    data.z,
    data.time,
    data.event,
    random_state=8,
    hazard_config=hazard,
    direction_config=dnn_direction,
    generator_config=generator,
    test_config=test,
)
dnn_result.as_dict()

{'p_value': 0.0002628285260793639,
 'log_p_value': -8.24400873051946,
 'test_statistic': 3.4673295244526217,
 'reject_null': True,
 'alpha': 0.05,
 'n_train': 200,
 'n_test': 200,
 'score_mean': 0.009352718493354582,
 'score_sd': 0.03814676754857265,
 'standardized_score_mean': 0.24517722193487756,
 'direction_loss': -6.716189384460449,
 'time_scale': 4.761494159698486,
 'random_state': 8}

## Optional random-forest direction estimator

Use `RandomForestDirectionConfig` to change only the estimator of $\phi$. The hazard and conditional-generator estimators remain unchanged.

In [4]:
forest_direction = RandomForestDirectionConfig(
    n_estimators=10,
    max_depth=2,
    min_samples_leaf=20,
    n_jobs=-1,
)

forest_result = scis_test(
    data.x,
    data.z,
    data.time,
    data.event,
    random_state=8,
    hazard_config=hazard,
    direction_config=forest_direction,
    generator_config=generator,
    test_config=test,
)
forest_result.as_dict()

{'p_value': 3.7918493777935053e-06,
 'log_p_value': -12.482656695406417,
 'test_statistic': 4.476611730465329,
 'reject_null': True,
 'alpha': 0.05,
 'n_train': 200,
 'n_test': 200,
 'score_mean': 0.04731605723878205,
 'score_sd': 0.14947691221403211,
 'standardized_score_mean': 0.3165442511351279,
 'direction_loss': -11.048698425292969,
 'time_scale': 4.761494159698486,
 'random_state': 8}

In [5]:
{
    "DNN": {
        "statistic": dnn_result.test_statistic,
        "p_value": dnn_result.p_value,
    },
    "random_forest": {
        "statistic": forest_result.test_statistic,
        "p_value": forest_result.p_value,
    },
}

{'DNN': {'statistic': 3.4673295244526217, 'p_value': 0.0002628285260793639},
 'random_forest': {'statistic': 4.476611730465329,
  'p_value': 3.7918493777935053e-06}}

## Aggregated scisTest

`K` is the number of disjoint subsamples formed in each permutation. Each subsample contains $\lfloor n/K\rfloor$ observations, and the number of calibration rows is $B=JK$. This example uses the package defaults $(K,J,L)=(3,5,6)$ and reuses the same DNN model-fitting configurations defined above.

In [6]:
aggregated_result = aggregated_scis_test(
    data.x,
    data.z,
    data.time,
    data.event,
    K=3,
    J=5,
    L=6,
    random_state=8,
    hazard_config=hazard,
    direction_config=dnn_direction,
    generator_config=generator,
    test_config=test,
)
aggregated_result.as_dict()

{'aggregated_p_value': 4.80090817501805e-07,
 'empirical_p_value': 0.0,
 'aggregate_statistic': 1.8422741569304735,
 'exceedance_count': 0,
 'empirical_resolution': 0.06666666666666667,
 'bandwidth': 0.25535565060735976,
 'n': 400,
 'K': 3,
 'subsample_size': 133,
 'J': 5,
 'L': 6,
 'B': 15}